<a href="https://colab.research.google.com/github/Ishan1923/Skin-Cancer-Detection-using-DL-techniques/blob/hybridModel/main/skincancerdetectionusingvgg16arch-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nodoubttome_skin_cancer9_classesisic_path = kagglehub.dataset_download('nodoubttome/skin-cancer9-classesisic')

print('Data source import complete.')


Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024371.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024358.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024337.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024382.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024420.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024409.jpg
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration

In [3]:
# !pip install tensorflow[and-cuda] keras matplotlib glob
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import glob
from keras import layers
from tensorflow import data as tf_data

In [4]:
# !pip install tensorflow-gpu
!nvidia-smi

Wed Jul 16 20:33:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
print("GPU available: ", tf.config.list_physical_devices('GPU'))

GPU available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# !pip install kagglehub
import kagglehub

# Download the dataset
dataset_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", dataset_path)

Path to dataset files: /kaggle/input/skin-cancer9-classesisic


In [7]:
print("current directory contents: \n", os.listdir(dataset_path))

current directory contents: 
 ['Skin cancer ISIC The International Skin Imaging Collaboration']


In [8]:
ISIC_folder = os.path.join(dataset_path, os.listdir(dataset_path)[0])
print(ISIC_folder)

/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration


In [9]:
print(os.listdir(ISIC_folder))

['Test', 'Train']


In [10]:
train_image_folder = os.path.join(ISIC_folder, os.listdir(ISIC_folder)[1])
print(train_image_folder)

/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train


In [11]:
os.listdir(train_image_folder)

['pigmented benign keratosis',
 'melanoma',
 'vascular lesion',
 'actinic keratosis',
 'squamous cell carcinoma',
 'basal cell carcinoma',
 'seborrheic keratosis',
 'dermatofibroma',
 'nevus']

In [12]:
test_image_folder = os.path.join(ISIC_folder, os.listdir(ISIC_folder)[0])
print(test_image_folder)

/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test


In [13]:
os.listdir(train_image_folder)

['pigmented benign keratosis',
 'melanoma',
 'vascular lesion',
 'actinic keratosis',
 'squamous cell carcinoma',
 'basal cell carcinoma',
 'seborrheic keratosis',
 'dermatofibroma',
 'nevus']

In [14]:
os.listdir(test_image_folder)

['pigmented benign keratosis',
 'melanoma',
 'vascular lesion',
 'actinic keratosis',
 'squamous cell carcinoma',
 'basal cell carcinoma',
 'seborrheic keratosis',
 'dermatofibroma',
 'nevus']

In [15]:
train_images_address = []
train_images_labels = []
test_images_address = []
test_images_labels = []

count = 0
for folder in os.listdir(train_image_folder):
    folder_path = os.path.join(train_image_folder, folder)
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        train_images_address.append(file_path)
        train_images_labels.append(count)
    count = count + 1

count = 0
for folder in os.listdir(test_image_folder):
    folder_path = os.path.join(test_image_folder, folder)
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        test_images_address.append(file_path)
        test_images_labels.append(count)
    count = count + 1

print("total classes in training folder: ", len(train_images_address))
print("total classes in test folder: ", len(test_images_address))
print("total training images: ",len(train_images_labels))
print("total test images", len(test_images_labels))



total classes in training folder:  2239
total classes in test folder:  118
total training images:  2239
total test images 118


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

class FullPreprocessingPipeline:
    def __init__(self, img_size=(224,224), batch_size=32):
        self.img_size = img_size
        self.batch_size = batch_size

    def resize_image(self, image):
        image = tf.image.resize(image, self.img_size)
        return image

    def enhance_image(self, image):
        def _enhance(img_np):
            img = (img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np
            # Histogram Equalization (Y channel)
            img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
            img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
            img_eq = cv2.cvtColor(img_yuv, cv2.COLOR_YCrCb2RGB)
            # Grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            return img_eq.astype(np.float32)/255.0, gray_3ch.astype(np.float32)/255.0
        eq_img, gray_img = tf.numpy_function(_enhance, [image], [tf.float32, tf.float32])
        eq_img.set_shape([self.img_size[0], self.img_size[1], 3])
        gray_img.set_shape([self.img_size[0], self.img_size[1], 3])
        return eq_img, gray_img

    def segment_image(self, image):
        def _segment(img_np):
            img = (img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            # Otsu thresholding
            _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
            mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
            masked = cv2.bitwise_and(img, mask_3ch)
            return masked.astype(np.float32)/255.0, mask.astype(np.float32)/255.0
        seg_img, mask = tf.numpy_function(_segment, [image], [tf.float32, tf.float32])
        seg_img.set_shape([self.img_size[0], self.img_size[1], 3])
        mask.set_shape([self.img_size[0], self.img_size[1]])
        return seg_img, mask

    def augment_image(self, image):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_flip_up_down(image)
        image = tf.image.random_brightness(image, max_delta=0.2)
        image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
        k = tf.random.uniform([], 0, 4, tf.int32)
        image = tf.image.rot90(image, k)
        image = tf.clip_by_value(image, 0, 1)
        return image

    def detect_hair(self, image):
        # Simple hair detection: count dark lines in grayscale
        def _hair(img_np):
            img = (img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            hair_pixels = np.sum(edges > 0)
            return np.array([hair_pixels > 500], dtype=np.bool_) # threshold
        has_hair = tf.numpy_function(_hair, [image], tf.bool)
        return has_hair

    def image_contrast(self, image):
        def _contrast(img_np):
            img = (img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            contrast = gray.std()
            return np.array([contrast], dtype=np.float32)
        contrast = tf.numpy_function(_contrast, [image], tf.float32)
        return contrast

    def preprocess_and_group(self, image_path, label) -> dict():
        image = tf.io.read_file(image_path)
        image = tf.image.decode_image(image, channels=3)
        image.set_shape([None, None, 3])
        image = tf.image.convert_image_dtype(image, tf.float32)
        image = self.resize_image(image)

        eq_img, gray_img = self.enhance_image(image)
        seg_img, mask = self.segment_image(image)
        aug_img = self.augment_image(image)

        has_hair = self.detect_hair(image)
        contrast = self.image_contrast(image)

        return {
            'original': image,
            'enhanced': eq_img,
            'grayscale': gray_img,
            'segmented': seg_img,
            'mask': mask,
            'augmented': aug_img,
            'label': label,
            'has_hair': has_hair,
            'contrast': contrast
        }

    def create_full_dataset(self, image_paths, labels):
        dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
        dataset = dataset.map(self.preprocess_and_group, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.cache().batch(self.batch_size).prefetch(tf.data.AUTOTUNE)
        return dataset

    def visualize_comparison(self, batch, num_images=3):
    # Visualize original, enhanced, grayscale, segmented, augmented, mask
        for i in range(num_images):
            plt.figure(figsize=(18,3))
            plt.subplot(1,6,1)
            plt.imshow(np.asarray(batch['original'][i]))
            plt.title('Original')
            plt.axis('off')
            plt.subplot(1,6,2)
            plt.imshow(np.asarray(batch['enhanced'][i]))
            plt.title('Enhanced')
            plt.axis('off')
            plt.subplot(1,6,3)
            plt.imshow(np.asarray(batch['grayscale'][i]))
            plt.title('Grayscale')
            plt.axis('off')
            plt.subplot(1,6,4)
            plt.imshow(np.asarray(batch['segmented'][i]))
            plt.title('Segmented')
            plt.axis('off')
            plt.subplot(1,6,5)
            plt.imshow(np.asarray(batch['augmented'][i]))
            plt.title('Augmented')
            plt.axis('off')
            plt.subplot(1,6,6)
            plt.imshow(np.asarray(batch['mask'][i]), cmap='gray')
            plt.title('Mask')
            plt.axis('off')
            plt.show()

    def print_batch_info(self, batch):
        for i in range(len(batch['original'])):
            print(f"Image {i+1}:")
            print(f"  Label: {batch['label'][i].numpy()}")
            print(f"  Hair present: {batch['has_hair'][i].numpy()}")
            print(f"  Contrast: {batch['contrast'][i].numpy().item():.2f}")

    @tf.autograph.experimental.do_not_convert
    def group_and_show(self, dataset):
        for batch in dataset.take(1):
            # Group by hair
            hair_idx = [i for i, h in enumerate(batch['has_hair'].numpy()) if h]
            no_hair_idx = [i for i, h in enumerate(batch['has_hair'].numpy()) if not h]
            print(f"\nImages with hair: {hair_idx}")
            print(f"Images without hair: {no_hair_idx}")

            # Group by contrast
            contrasts = batch['contrast'].numpy()
            low_contrast_idx = [i for i, c in enumerate(contrasts) if c < 40]
            high_contrast_idx = [i for i, c in enumerate(contrasts) if c >= 40]
            print(f"Low contrast images: {low_contrast_idx}")
            print(f"High contrast images: {high_contrast_idx}")

            # Visualize a few from each group
            print("\n--- Visualizing images with hair ---")
            if hair_idx:
                self.visualize_comparison({k: v.numpy()[hair_idx] for k, v in batch.items()}, num_images=min(3, len(hair_idx)))
            print("\n--- Visualizing low contrast images ---")
            if low_contrast_idx:
                self.visualize_comparison({k: v.numpy()[low_contrast_idx] for k, v in batch.items()}, num_images=min(3, len(low_contrast_idx)))
            print("\n--- Visualizing high contrast images ---")
            if high_contrast_idx:
                self.visualize_comparison({k: v.numpy()[high_contrast_idx] for k, v in batch.items()}, num_images=min(3, len(high_contrast_idx)))

            # Print info for all images in batch
            print("\n--- Batch Info ---")
            self.print_batch_info(batch)


    def dataset_split(self, dataset, split_ratio=0.8, shuffle=True, seed=42):
        """
        Split a TensorFlow dataset into two datasets based on the given ratio.

        Args:
            dataset: tf.data.Dataset - The dataset to split
            split_ratio: float - Ratio for the first dataset (default: 0.8 for 80% train, 20% val)
            shuffle: bool - Whether to shuffle before splitting (default: True)
            seed: int - Random seed for reproducibility (default: 42)

        Returns:
            tuple: (first_dataset, second_dataset)
        """
        if not 0 < split_ratio < 1:
            raise ValueError("split_ratio must be between 0 and 1")

        # Get the total number of batches
        total_batches = tf.data.experimental.cardinality(dataset).numpy()

        if total_batches == tf.data.experimental.INFINITE_CARDINALITY:
            raise ValueError("Cannot split infinite dataset. Make sure dataset is finite.")

        # Calculate split point
        train_size = int(total_batches * split_ratio)

        # Shuffle if requested
        if shuffle:
            dataset = dataset.shuffle(buffer_size=total_batches, seed=seed, reshuffle_each_iteration=False)

        # Split the dataset
        first_dataset = dataset.take(train_size)
        second_dataset = dataset.skip(train_size)

        return first_dataset, second_dataset

# Usage Example:
pipeline = FullPreprocessingPipeline(img_size=(224,224), batch_size=16)
full_dataset = pipeline.create_full_dataset(train_images_address, train_images_labels)
train_dataset, val_dataset = pipeline.dataset_split(full_dataset)
test_dataset = pipeline.create_full_dataset(test_images_address, test_images_labels)
for batch in train_dataset.take(1):
    pipeline.visualize_comparison(batch)
    pipeline.print_batch_info(batch)
pipeline.group_and_show(train_dataset)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name 'fscope' is not defined
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name 'fscope' is not defined
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, cohen_kappa_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc
)
from sklearn.preprocessing import LabelBinarizer
import xgboost as xgb
import lightgbm as lgb
import itertools
import tensorflow as tf
from keras.applications import MobileNetV2
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D, Dropout

class ComprehensiveXGBoostLightGBM:
    def __init__(self, feature_extractor_type='mobilenet', img_size=(224, 224)):
        self.img_size = img_size
        self.feature_extractor = self._build_feature_extractor(feature_extractor_type)
        self.models = {}
        self.results = {}
        self.class_names = None

    def _build_feature_extractor(self, model_type):
        """Build feature extraction model"""
        if model_type == 'mobilenet':
            return MobileNetV2(weights='imagenet', include_top=False, pooling='avg',
                             input_shape=self.img_size + (3,))
        # Add other extractors as needed

    def extract_features_from_dataset(self, dataset):
        """Extract features from your preprocessed dataset"""
        features = []
        labels = []

        for batch in dataset:
            # Extract features from original images
            batch_features = self.feature_extractor.predict(batch['original'], verbose=0)
            features.append(batch_features)
            labels.append(batch['label'].numpy())

        features = np.vstack(features)
        labels = np.concatenate(labels)

        return features, labels

    def calculate_iou_multiclass(self, y_true, y_pred, num_classes):
        """Calculate IoU for multiclass classification"""
        ious = []

        for class_id in range(num_classes):
            # Convert to binary problem for each class
            true_binary = (y_true == class_id).astype(int)
            pred_binary = (y_pred == class_id).astype(int)

            # Calculate IoU for this class
            intersection = np.sum(true_binary & pred_binary)
            union = np.sum(true_binary | pred_binary)

            if union == 0:
                iou = 1.0 if intersection == 0 else 0.0
            else:
                iou = intersection / union

            ious.append(iou)

        return np.array(ious)

    def calculate_comprehensive_metrics(self, y_true, y_pred, y_prob=None, model_name="Model"):
        """Calculate all comprehensive metrics"""
        num_classes = len(np.unique(y_true))

        # Basic metrics
        accuracy = accuracy_score(y_true, y_pred)
        precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
        precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
        recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
        recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        # Kappa score
        kappa = cohen_kappa_score(y_true, y_pred)

        # IoU
        iou_per_class = self.calculate_iou_multiclass(y_true, y_pred, num_classes)
        mean_iou = np.mean(iou_per_class)

        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)

        # Per-class metrics
        per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
        per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
        per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

        # AUC if probabilities are provided
        auc_scores = None
        if y_prob is not None and num_classes > 2:
            # For multiclass, calculate AUC for each class vs rest
            lb = LabelBinarizer()
            y_true_binary = lb.fit_transform(y_true)
            if y_true_binary.shape[1] == 1:  # Binary case
                y_true_binary = np.hstack([1 - y_true_binary, y_true_binary])

            auc_scores = []
            for i in range(num_classes):
                if len(np.unique(y_true_binary[:, i])) > 1:  # Check if class exists
                    auc_score = roc_auc_score(y_true_binary[:, i], y_prob[:, i])
                    auc_scores.append(auc_score)
                else:
                    auc_scores.append(0.0)
            auc_scores = np.array(auc_scores)

        # Store results
        results = {
            'model_name': model_name,
            'accuracy': accuracy,
            'precision_macro': precision_macro,
            'precision_micro': precision_micro,
            'recall_macro': recall_macro,
            'recall_micro': recall_micro,
            'f1_macro': f1_macro,
            'f1_micro': f1_micro,
            'f1_weighted': f1_weighted,
            'kappa': kappa,
            'mean_iou': mean_iou,
            'iou_per_class': iou_per_class,
            'confusion_matrix': cm,
            'per_class_precision': per_class_precision,
            'per_class_recall': per_class_recall,
            'per_class_f1': per_class_f1,
            'auc_scores': auc_scores,
            'y_true': y_true,
            'y_pred': y_pred,
            'y_prob': y_prob
        }

        return results

    def train_xgboost(self, X_train, y_train, X_val, y_val, model_name="XGBoost"):
        """Train XGBoost with comprehensive evaluation"""
        print(f"Training {model_name}...")

        # XGBoost parameters
        params = {
            'objective': 'multi:softprob',
            'num_class': len(np.unique(y_train)),
            'max_depth': 6,
            'learning_rate': 0.1,
            'n_estimators': 1000,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': 42,
            'early_stopping_rounds': 50,
            'eval_metric': 'mlogloss'
        }

        model = xgb.XGBClassifier(**params)

        # Train with validation
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=100
        )

        # Predictions
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)

        # Calculate metrics
        results = self.calculate_comprehensive_metrics(y_val, y_pred, y_prob, model_name)

        self.models[model_name] = model
        self.results[model_name] = results

        return model, results

    def train_lightgbm(self, X_train, y_train, X_val, y_val, model_name="LightGBM"):
        """Train LightGBM with comprehensive evaluation"""
        print(f"Training {model_name}...")

        # LightGBM parameters
        params = {
            'objective': 'multiclass',
            'num_class': len(np.unique(y_train)),
            'max_depth': 6,
            'learning_rate': 0.1,
            'n_estimators': 1000,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': 42,
            'early_stopping_rounds': 50,
            'eval_metric': 'multi_logloss'
        }

        model = lgb.LGBMClassifier(**params)

        # Train with validation
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
            # verbose=100  # Removed verbose as it's not a supported argument in this version
        )

        # Predictions
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)

        # Calculate metrics
        results = self.calculate_comprehensive_metrics(y_val, y_pred, y_prob, model_name)

        self.models[model_name] = model
        self.results[model_name] = results

        return model, results

    def plot_confusion_matrix(self, results, class_names=None, figsize=(8, 6)):
        """Plot confusion matrix"""
        cm = results['confusion_matrix']

        plt.figure(figsize=figsize)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {results["model_name"]}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()

    def plot_classification_metrics(self, results, class_names=None, figsize=(12, 8)):
        """Plot comprehensive classification metrics"""
        fig, axes = plt.subplots(2, 2, figsize=figsize)

        # Per-class metrics
        classes = class_names if class_names else [f'Class {i}' for i in range(len(results['per_class_f1']))]

        # F1 Score per class
        axes[0, 0].bar(classes, results['per_class_f1'])
        axes[0, 0].set_title('F1 Score per Class')
        axes[0, 0].set_ylabel('F1 Score')
        axes[0, 0].tick_params(axis='x', rotation=45)

        # Precision per class
        axes[0, 1].bar(classes, results['per_class_precision'])
        axes[0, 1].set_title('Precision per Class')
        axes[0, 1].set_ylabel('Precision')
        axes[0, 1].tick_params(axis='x', rotation=45)

        # Recall per class
        axes[1, 0].bar(classes, results['per_class_recall'])
        axes[1, 0].set_title('Recall per Class')
        axes[1, 0].set_ylabel('Recall')
        axes[1, 0].tick_params(axis='x', rotation=45)

        # IoU per class
        axes[1, 1].bar(classes, results['iou_per_class'])
        axes[1, 1].set_title('IoU per Class')
        axes[1, 1].set_ylabel('IoU')
        axes[1, 1].tick_params(axis='x', rotation=45)

        plt.suptitle(f'Classification Metrics - {results["model_name"]}')
        plt.tight_layout()
        plt.show()

    def plot_roc_curves(self, results, class_names=None, figsize=(10, 8)):
        """Plot ROC curves for each class"""
        if results['y_prob'] is None:
            print("No probability predictions available for ROC curves")
            return

        num_classes = len(np.unique(results['y_true']))
        classes = class_names if class_names else [f'Class {i}' for i in range(num_classes)]

        plt.figure(figsize=figsize)

        # Binarize labels
        lb = LabelBinarizer()
        y_true_binary = lb.fit_transform(results['y_true'])
        if y_true_binary.shape[1] == 1:  # Binary case
            y_true_binary = np.hstack([1 - y_true_binary, y_true_binary])

        colors = plt.cm.Set1(np.linspace(0, 1, num_classes))

        for i, color in zip(range(num_classes), colors):
            if len(np.unique(y_true_binary[:, i])) > 1:
                fpr, tpr, _ = roc_curve(y_true_binary[:, i], results['y_prob'][:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, color=color, lw=2,
                        label=f'{classes[i]} (AUC = {roc_auc:.2f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves - {results["model_name"]}')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_metrics_comparison(self, figsize=(15, 10)):
        """Plot comparison of all metrics between models"""
        if len(self.results) < 2:
            print("Need at least 2 models for comparison")
            return

        fig, axes = plt.subplots(2, 3, figsize=figsize)

        model_names = list(self.results.keys())

        # Overall metrics
        metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'kappa', 'mean_iou']
        metric_titles = ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 (Macro)', 'Kappa Score', 'Mean IoU']

        for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
            row = idx // 3
            col = idx % 3

            values = [self.results[model][metric] for model in model_names]
            bars = axes[row, col].bar(model_names, values, color=['skyblue', 'lightcoral'][:len(model_names)])
            axes[row, col].set_title(title)
            axes[row, col].set_ylabel('Score')
            axes[row, col].set_ylim(0, 1)

            # Add value labels on bars
            for bar, value in zip(bars, values):
                axes[row, col].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                                  f'{value:.3f}', ha='center', va='bottom')

        plt.suptitle('Model Comparison - All Metrics')
        plt.tight_layout()
        plt.show()

    def print_comprehensive_report(self, results):
        """Print detailed classification report"""
        print(f"\n{'='*60}")
        print(f"COMPREHENSIVE REPORT - {results['model_name']}")
        print(f"{'='*60}")

        print(f"Accuracy: {results['accuracy']:.4f}")
        print(f"Precision (Macro): {results['precision_macro']:.4f}")
        print(f"Precision (Micro): {results['precision_micro']:.4f}")
        print(f"Recall (Macro): {results['recall_macro']:.4f}")
        print(f"Recall (Micro): {results['recall_micro']:.4f}")
        print(f"F1 Score (Macro): {results['f1_macro']:.4f}")
        print(f"F1 Score (Micro): {results['f1_micro']:.4f}")
        print(f"F1 Score (Weighted): {results['f1_weighted']:.4f}")
        print(f"Kappa Score: {results['kappa']:.4f}")
        print(f"Mean IoU: {results['mean_iou']:.4f}")

        if results['auc_scores'] is not None:
            print(f"Mean AUC: {np.mean(results['auc_scores']):.4f}")

        print(f"\nPer-Class Metrics:")
        for i in range(len(results['per_class_f1'])):
            print(f"  Class {i}: Precision={results['per_class_precision'][i]:.4f}, "
                  f"Recall={results['per_class_recall'][i]:.4f}, "
                  f"F1={results['per_class_f1'][i]:.4f}, "
                  f"IoU={results['iou_per_class'][i]:.4f}")

        print(f"\nConfusion Matrix:")
        print(results['confusion_matrix'])

    def run_complete_analysis(self, train_dataset, val_dataset, test_dataset, class_names=None):
        """Run complete analysis with both XGBoost and LightGBM"""
        self.class_names = class_names

        # Extract features
        print("Extracting features...")
        X_train, y_train = self.extract_features_from_dataset(train_dataset)
        X_val, y_val = self.extract_features_from_dataset(val_dataset)
        X_test, y_test = self.extract_features_from_dataset(test_dataset)

        print(f"Feature shape: {X_train.shape}")
        print(f"Number of classes: {len(np.unique(y_train))}")

        # Train both models
        xgb_model, xgb_results = self.train_xgboost(X_train, y_train, X_val, y_val)
        lgb_model, lgb_results = self.train_lightgbm(X_train, y_train, X_val, y_val)

        # Test on test set
        print("\nEvaluating on test set...")
        for model_name, model in self.models.items():
            y_pred_test = model.predict(X_test)
            y_prob_test = model.predict_proba(X_test)

            test_results = self.calculate_comprehensive_metrics(
                y_test, y_pred_test, y_prob_test, f"{model_name}_Test"
            )

            self.results[f"{model_name}_Test"] = test_results

            # Print comprehensive report
            self.print_comprehensive_report(test_results)

            # Plot all visualizations
            self.plot_confusion_matrix(test_results, class_names)
            self.plot_classification_metrics(test_results, class_names)
            self.plot_roc_curves(test_results, class_names)

        # Compare models
        test_results_only = {k: v for k, v in self.results.items() if "_Test" in k}
        if len(test_results_only) >= 2:
            # Create temporary results for comparison
            temp_results = self.results.copy()
            self.results = test_results_only
            self.plot_metrics_comparison()
            self.results = temp_results

        return self.models, self.results

# Usage Example:
# Initialize the comprehensive analyzer
analyzer = ComprehensiveXGBoostLightGBM(
    feature_extractor_type='mobilenet',
    img_size=(224, 224)
)

# Define your class names (optional)
class_names = os.listdir(train_image_folder)  # Replace with your actual class names

# Run complete analysis
models, results = analyzer.run_complete_analysis(
    train_dataset,
    val_dataset,
    test_dataset,
    class_names=class_names
)

# Access individual results
xgb_test_results = results['XGBoost_Test']
lgb_test_results = results['LightGBM_Test']

# You can also train individual models
# xgb_model, xgb_results = analyzer.train_xgboost(X_train, y_train, X_val, y_val)
# analyzer.print_comprehensive_report(xgb_results)
# analyzer.plot_confusion_matrix(xgb_results, class_names)

Here are the arguments used in the `ComprehensiveXGBoostLightGBM` initialization and the `run_complete_analysis` method:

In [ ]:
# Select an image to visualize features for
sample_image_path = test_images_address[0] # You can choose any image here

# Preprocess the image (without augmentation)
single_image_dataset_viz = tf.data.Dataset.from_tensor_slices(([sample_image_path], [0])) # Label doesn't matter for visualization
single_image_dataset_viz = single_image_dataset_viz.map(pipeline.preprocess_and_group, num_parallel_calls=tf.data.AUTOTUNE)
single_image_dataset_viz = single_image_dataset_viz.batch(1).prefetch(tf.data.AUTOTUNE)

for batch in single_image_dataset_viz.take(1):
    original_image_viz = np.asarray(batch['original'][0])
    input_image = batch['original']


# Create a model that outputs the activations of intermediate layers
layer_outputs = [analyzer.feature_extractor.get_layer(layer.name).output for layer in analyzer.feature_extractor.layers if 'conv' in layer.name or 'block' in layer.name] # Select layers to visualize
feature_map_model = tf.keras.models.Model(inputs=analyzer.feature_extractor.inputs, outputs=layer_outputs)

# Get the feature maps for the sample image
feature_maps = feature_map_model.predict(input_image)

# Visualize the feature maps
print("Visualizing feature maps for selected layers:")

for layer_name, feature_map in zip([layer.name for layer in analyzer.feature_extractor.layers if 'conv' in layer.name or 'block' in layer.name], feature_maps):
    print(f"Layer: {layer_name}, Shape: {feature_map.shape}")

    # We will visualize a subset of the feature maps for each layer
    num_filters_to_show = min(32, feature_map.shape[-1]) # Show up to 32 filters
    fig, axes = plt.subplots(1, num_filters_to_show, figsize=(20, 2))

    for i in range(num_filters_to_show):
        ax = axes[i]
        ax.imshow(feature_map[0, :, :, i], cmap='viridis')
        ax.set_title(f'Filter {i}')
        ax.axis('off')

    plt.suptitle(f'Feature Maps for Layer: {layer_name}', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
from PIL import Image

def analyze_image_properties(image_paths):
    """Analyzes image properties (size, dimensions, color histogram)."""
    sizes = []
    widths = []
    heights = []
    # Initialize histogram bins for RGB (0-255)
    histograms = {
        'R': np.zeros(256, dtype=int),
        'G': np.zeros(256, dtype=int),
        'B': np.zeros(256, dtype=int)
    }

    for img_path in image_paths:
        try:
            with Image.open(img_path) as img:
                sizes.append(os.path.getsize(img_path))
                widths.append(img.width)
                heights.append(img.height)

                # Calculate histogram for each channel
                if img.mode == 'RGB':
                    hist = img.histogram()
                    # hist is a list like [R_bins..., G_bins..., B_bins...]
                    histograms['R'] += hist[:256]
                    histograms['G'] += hist[256:512]
                    histograms['B'] += hist[512:768]
                elif img.mode == 'L': # Grayscale
                     hist = img.histogram()
                     # For grayscale, add to all R, G, B for simplicity in combined plot
                     histograms['R'] += hist
                     histograms['G'] += hist
                     histograms['B'] += hist


        except Exception as e:
            print(f"Could not process image {img_path}: {e}")
            continue

    return sizes, widths, heights, histograms

# Analyze training and test images
print("Analyzing training images...")
train_sizes, train_widths, train_heights, train_histograms = analyze_image_properties(train_images_address)

print("Analyzing test images...")
test_sizes, test_widths, test_heights, test_histograms = analyze_image_properties(test_images_address)

In [ ]:
# Visualize image size distribution
plt.figure(figsize=(12, 6))
sns.histplot(train_sizes, kde=True, color='skyblue', label='Train')
sns.histplot(test_sizes, kde=True, color='lightcoral', label='Test')
plt.title('Distribution of Image File Sizes')
plt.xlabel('File Size (bytes)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# Visualize image dimensions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(train_widths, kde=True, color='skyblue', ax=axes[0], label='Train')
sns.histplot(test_widths, kde=True, color='lightcoral', ax=axes[0], label='Test')
axes[0].set_title('Distribution of Image Widths')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.histplot(train_heights, kde=True, color='skyblue', ax=axes[1], label='Train')
sns.histplot(test_heights, kde=True, color='lightcoral', ax=axes[1], label='Test')
axes[1].set_title('Distribution of Image Heights')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()


# Visualize pixel value histograms
plt.figure(figsize=(12, 6))
plt.plot(train_histograms['R'], color='red', label='Train R')
plt.plot(train_histograms['G'], color='green', label='Train G')
plt.plot(train_histograms['B'], color='blue', label='Train B')

plt.plot(test_histograms['R'], color='salmon', linestyle='--', label='Test R')
plt.plot(test_histograms['G'], color='lightgreen', linestyle='--', label='Test G')
plt.plot(test_histograms['B'], color='lightblue', linestyle='--', label='Test B')

plt.title('Pixel Value Histograms (RGB Channels)')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.legend()
plt.show()